# Grainology State-wise AI Forecaster

This is a fully standalone Kaggle notebook. It does not import local project files such as `train.py`.

All model code is embedded directly into notebook cells, while the final release contract stays stable for the Grainology website.

Release output:

```text
/kaggle/working/release/
/kaggle/working/*.json, *.csv, *.parquet
/kaggle/working/grainology_release.zip
```


In [ ]:
%pip install -q "catboost>=1.2" "lightgbm>=4.0" "xgboost>=2.0" "optuna>=3.6" "jsonschema>=4.21" "matplotlib>=3.8" "numpy>=1.26" "pandas>=2.2" "polars>=1.0" "pyarrow>=15" "requests>=2.31" "scikit-learn>=1.4" "supabase>=2.0"


## Runtime Notes

Safe model-improvement knobs can be set as Kaggle environment variables:

- `ENABLE_OPTUNA_TUNING=true`
- `OPTUNA_TRIALS=25`
- `MAX_TRAIN_ROWS_PER_MODEL=250000`
- `ENSEMBLE_PRUNE_RATIO=1.08`
- `MIN_MAPE_IMPROVEMENT=0.01`

Keep `schema_version = 2.0`, release filenames, grains, horizons, and state-wise structure unchanged unless the website is migrated.


## Source: config

In [ ]:
from __future__ import annotations

import os
from pathlib import Path


def env_bool(name: str, default: bool = False) -> bool:
    return str(os.environ.get(name, str(default))).strip().lower() in {"1", "true", "yes", "y"}


INPUT_ROOT = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
STAGING_DIR = WORK_ROOT / "release_staging"
RELEASE_DIR = WORK_ROOT / "release"
MODEL_DIR = STAGING_DIR / "models"

CANONICAL_PARQUET = STAGING_DIR / "canonical_daily.parquet"
CANONICAL_CSV = STAGING_DIR / "canonical_daily.csv"

TARGET_GRAINS = ["Wheat", "Paddy", "Maize", "Mustard"]
HORIZONS = [7, 30, 90]
CANONICAL_SCHEMA_VERSION = "2.0"
RELEASE_SCHEMA_VERSION = "2.0"
MODEL_MODE = "global_state_aware"

AGGREGATION_METHOD = os.environ.get("AGGREGATION_METHOD", "median").strip().lower() or "median"
FORCE_FULL_REBUILD = env_bool("FORCE_FULL_REBUILD", False)
FORCE_RETRAIN = env_bool("FORCE_RETRAIN", False)
AI_PREDICTION_BUCKET = os.environ.get("AI_PREDICTION_BUCKET", "ai-predictions")
MIN_STATE_OBSERVED_DAYS = int(os.environ.get("MIN_STATE_OBSERVED_DAYS", "240"))
MIN_VALIDATION_SAMPLES = int(os.environ.get("MIN_VALIDATION_SAMPLES", "20"))
MIN_MAPE_IMPROVEMENT = float(os.environ.get("MIN_MAPE_IMPROVEMENT", "0.01"))
FORECAST_HISTORY_DAYS = int(os.environ.get("FORECAST_HISTORY_DAYS", "365"))
EFFICIENCY_MAX_ROWS_PER_SERIES = int(os.environ.get("EFFICIENCY_MAX_ROWS_PER_SERIES", "0"))
MAX_TRAIN_ROWS_PER_MODEL = int(os.environ.get("MAX_TRAIN_ROWS_PER_MODEL", "250000"))
ENSEMBLE_PRUNE_RATIO = float(os.environ.get("ENSEMBLE_PRUNE_RATIO", "1.08"))
ENABLE_OPTUNA_TUNING = env_bool("ENABLE_OPTUNA_TUNING", False)
OPTUNA_TRIALS = int(os.environ.get("OPTUNA_TRIALS", "25"))
OPTUNA_TIMEOUT_SECONDS = int(os.environ.get("OPTUNA_TIMEOUT_SECONDS", "180"))

REQUIRED_RELEASE_FILES = [
    "manifest.json",
    "predictions.json",
    "actuals.json",
    "forecast_series.json",
    "historical_efficiency.json",
    "backtest.json",
    "reasoning.json",
    "states.json",
    "metrics.json",
    "canonical_daily.parquet",
    "canonical_daily.csv",
    "checksums.json",
]


## Source: historical_loader

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd


COLUMN_ALIASES = {
    "date": ["date", "arrival_date", "price_date", "reported_date"],
    "state_name": ["state", "state_name", "state_name_en"],
    "district": ["district", "district_name"],
    "market": ["market", "market_name"],
    "grain": ["commodity", "cmdt_name", "commodity_name", "grain"],
    "variety": ["variety", "variety_name"],
    "grade": ["grade"],
    "price": ["modal_price", "modalprice", "price", "modal_price_rs_quintal"],
    "price_low": ["min_price", "minprice", "price_low"],
    "price_high": ["max_price", "maxprice", "price_high"],
    "arrival": ["arrival", "arrival_qty", "arrival_metric_tonnes", "arrival_mt"],
}


def normalize_grain(value: object) -> str | None:
    text = str(value or "").strip().lower()
    if not text:
        return None
    if "mustard oil" in text or "oil" in text and "mustard" in text:
        return None
    if "wheat" in text:
        return "Wheat"
    if "paddy" in text:
        return "Paddy"
    if "maize" in text or "corn" in text:
        return "Maize"
    if "mustard" in text or "sarson" in text or "rapeseed" in text or "rape seed" in text:
        return "Mustard"
    return None


def discover_data_files(root: Path = INPUT_ROOT) -> list[Path]:
    parquet_files = sorted(path for path in root.rglob("*.parquet") if path.is_file())
    csv_files = sorted(path for path in root.rglob("*.csv") if path.is_file())
    yearly_parquets = [path for path in parquet_files if path.stem.isdigit()]
    latest_csvs = [path for path in csv_files if path.name.lower() == "latest_data.csv"]
    yearly_csvs = [path for path in csv_files if path.stem.isdigit()]
    if yearly_parquets:
        return yearly_parquets + latest_csvs
    if yearly_csvs:
        return yearly_csvs + latest_csvs
    if parquet_files:
        return parquet_files + latest_csvs
    return csv_files


def resolve_columns(columns: Iterable[str]) -> dict[str, str | None]:
    by_lower = {column.lower().strip(): column for column in columns}
    resolved: dict[str, str | None] = {}
    for target, aliases in COLUMN_ALIASES.items():
        resolved[target] = next((by_lower[alias] for alias in aliases if alias in by_lower), None)
    return resolved


def normalize_frame(df: pd.DataFrame, source: str, source_priority: int) -> pd.DataFrame:
    resolved = resolve_columns(df.columns)
    required = ["date", "state_name", "grain", "price"]
    if any(resolved[column] is None for column in required):
        return pd.DataFrame()

    out = pd.DataFrame()
    for column, source_column in resolved.items():
        if source_column is None:
            out[column] = np.nan
        else:
            out[column] = df[source_column]

    out["date"] = pd.to_datetime(out["date"], errors="coerce", dayfirst=True).dt.date.astype("string")
    out["state_name"] = out["state_name"].astype("string").str.strip()
    out["grain"] = out["grain"].map(normalize_grain)
    for column in ["price", "price_low", "price_high", "arrival"]:
        out[column] = pd.to_numeric(out[column], errors="coerce")

    out = out[
        out["date"].notna()
        & out["state_name"].notna()
        & out["grain"].isin(TARGET_GRAINS)
        & out["price"].between(100, 20000)
    ].copy()

    out["market_count"] = 1
    out["state_id"] = pd.NA
    out["state_key"] = out["state_name"].str.lower().str.replace(r"[^a-z0-9]+", "-", regex=True).str.strip("-")
    out["is_observed"] = True
    out["source"] = source
    out["source_priority"] = source_priority
    out["source_fetched_at"] = pd.NaT
    return out


def aggregate_daily(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    price_agg = "mean" if AGGREGATION_METHOD == "mean" else "median"
    grouped = df.groupby(["date", "state_name", "grain"], as_index=False).agg(
        state_id=("state_id", "first"),
        state_key=("state_key", "first"),
        price=("price", price_agg),
        price_low=("price_low", "min"),
        price_high=("price_high", "max"),
        arrival=("arrival", "sum"),
        market_count=("market_count", "sum"),
        is_observed=("is_observed", "max"),
        source=("source", "last"),
        source_priority=("source_priority", "max"),
        source_fetched_at=("source_fetched_at", "max"),
    )

    all_states = df.groupby(["date", "grain"], as_index=False).agg(
        price=("price", price_agg),
        price_low=("price_low", "min"),
        price_high=("price_high", "max"),
        arrival=("arrival", "sum"),
        market_count=("market_count", "sum"),
        is_observed=("is_observed", "max"),
        source=("source", "last"),
        source_priority=("source_priority", "max"),
        source_fetched_at=("source_fetched_at", "max"),
    )
    all_states["state_name"] = "All States"
    all_states["state_id"] = "100006"
    all_states["state_key"] = "all-states"

    return pd.concat([grouped, all_states[grouped.columns]], ignore_index=True)


def load_path(path: Path, source_priority: int) -> pd.DataFrame:
    try:
        if path.suffix.lower() == ".parquet":
            df = pd.read_parquet(path)
        else:
            df = pd.read_csv(path)
    except Exception as exc:
        print(f"Skipping {path}: {exc}")
        return pd.DataFrame()
    source = "latest_csv" if path.name.lower() == "latest_data.csv" else "historical"
    return normalize_frame(df, source=source, source_priority=source_priority)


def load_historical_sources() -> pd.DataFrame:
    files = discover_data_files()
    if not files:
        print(f"No historical files found under {INPUT_ROOT}")
        return pd.DataFrame()

    frames = []
    for path in files:
        priority = 2 if path.name.lower() == "latest_data.csv" else 1
        frame = load_path(path, priority)
        if not frame.empty:
            frames.append(frame)

    if not frames:
        return pd.DataFrame()

    raw = pd.concat(frames, ignore_index=True)
    daily = aggregate_daily(raw)
    print(f"Loaded {len(daily):,} canonical historical rows from {len(files)} files")
    return daily


## Source: supabase_source

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from supabase import create_client



def read_secret(name: str) -> str | None:
    value = os.environ.get(name)
    if value:
        return value.strip()
    try:
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(name)
        return value.strip() if value else None
    except Exception:
        return None


def get_supabase_client():
    url = read_secret("SUPABASE_URL")
    key = read_secret("SUPABASE_SECRET_KEY") or read_secret("SUPABASE_SERVICE_ROLE_KEY")
    if not url or not key:
        print("Supabase secrets are missing; continuing without live actual deltas")
        return None
    return create_client(url, key)


def get_supabase_actuals_watermark(client) -> str | None:
    try:
        response = (
            client.table("agmarknet_ai_actuals")
            .select("date")
            .order("date", desc=True)
            .limit(1)
            .execute()
        )
        rows = response.data or []
        return rows[0].get("date") if rows else None
    except Exception as exc:
        print(f"Could not read Supabase actuals watermark: {exc}")
        return None


def download_previous_canonical(destination: Path) -> bool:
    client = get_supabase_client()
    if not client:
        return False
    for object_path in [
        "canonical/latest/canonical_daily.parquet",
        "canonical/latest/canonical_daily.csv",
    ]:
        try:
            payload = client.storage.from_(AI_PREDICTION_BUCKET).download(object_path)
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(payload if isinstance(payload, bytes) else payload.read())
            print(f"Downloaded previous canonical snapshot: {object_path}")
            return True
        except Exception as exc:
            print(f"Previous canonical download skipped for {object_path}: {exc}")
    return False


def fetch_supabase_actuals(canonical_latest_date: str | None) -> pd.DataFrame:
    client = get_supabase_client()
    if not client:
        return pd.DataFrame()

    watermark = get_supabase_actuals_watermark(client)
    print(f"Supabase agmarknet_ai_actuals latest date: {watermark or 'none'}")
    print(f"Fetching Supabase actual rows newer than canonical date: {canonical_latest_date or 'none'}")

    rows = []
    page_size = 1000
    offset = 0
    while True:
        query = (
            client.table("agmarknet_ai_actuals")
            .select("*")
            .order("date", desc=False)
            .range(offset, offset + page_size - 1)
        )
        if canonical_latest_date:
            query = query.gt("date", canonical_latest_date)
        response = query.execute()
        batch = response.data or []
        rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size

    if not rows:
        print("Fetched 0 Supabase actual rows")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df = df[df["grain"].isin(TARGET_GRAINS)].copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date.astype("string")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df[df["price"].gt(0)]
    df["is_observed"] = True
    df["source"] = "supabase_agmarknet"
    df["source_priority"] = 3
    if df.empty:
        print("Fetched Supabase rows, but none survived grain/date/price filtering")
        return df

    print(
        f"Fetched {len(df):,} Supabase actual rows "
        f"from {df['date'].min()} to {df['date'].max()}"
    )
    return df


## Source: canonical_dataset

In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path

import pandas as pd


CANONICAL_COLUMNS = [
    "date",
    "state_name",
    "state_id",
    "state_key",
    "grain",
    "price",
    "price_low",
    "price_high",
    "arrival",
    "market_count",
    "is_observed",
    "source",
    "source_priority",
    "source_fetched_at",
]


def align_canonical(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=CANONICAL_COLUMNS)
    out = df.copy()
    for column in CANONICAL_COLUMNS:
        if column not in out.columns:
            out[column] = pd.NA
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.date.astype("string")
    out["price"] = pd.to_numeric(out["price"], errors="coerce")
    out["source_priority"] = pd.to_numeric(out["source_priority"], errors="coerce").fillna(1).astype(int)
    out["market_count"] = pd.to_numeric(out["market_count"], errors="coerce").fillna(0).astype(int)
    out["is_observed"] = out["is_observed"].fillna(True).astype(bool)
    return out[CANONICAL_COLUMNS].dropna(subset=["date", "state_name", "grain", "price"])


def load_previous_canonical() -> pd.DataFrame:
    previous = STAGING_DIR / "previous_canonical.parquet"
    if FORCE_FULL_REBUILD:
        return pd.DataFrame()
    if not previous.exists():
        download_previous_canonical(previous)
    if previous.exists():
        try:
            return align_canonical(pd.read_parquet(previous))
        except Exception:
            return align_canonical(pd.read_csv(previous))
    return pd.DataFrame()


def merge_priority(frames: list[pd.DataFrame]) -> pd.DataFrame:
    aligned = [align_canonical(frame) for frame in frames if frame is not None and not frame.empty]
    if not aligned:
        return pd.DataFrame(columns=CANONICAL_COLUMNS)
    merged = pd.concat(aligned, ignore_index=True)
    merged = merged.sort_values(["date", "state_name", "grain", "source_priority"])
    merged = merged.drop_duplicates(["date", "state_name", "grain"], keep="last")
    return merged.sort_values(["grain", "state_name", "date"]).reset_index(drop=True)


def validate_canonical(df: pd.DataFrame) -> None:
    if df.empty:
        raise ValueError("Canonical dataset is empty")
    duplicate_count = df.duplicated(["date", "state_name", "grain"]).sum()
    if duplicate_count:
        raise ValueError(f"Canonical dataset has {duplicate_count} duplicate keys")
    if df["price"].le(0).any():
        raise ValueError("Canonical dataset contains nonpositive prices")
    if "All States" not in set(df["state_name"]):
        raise ValueError("Canonical dataset does not contain All States rows")


def save_canonical(df: pd.DataFrame) -> None:
    STAGING_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(CANONICAL_PARQUET, index=False)
    df.to_csv(CANONICAL_CSV, index=False)
    shutil.copy(CANONICAL_PARQUET, Path("/kaggle/working/canonical_daily.parquet"))
    shutil.copy(CANONICAL_CSV, Path("/kaggle/working/canonical_daily.csv"))


def build_canonical_dataset() -> pd.DataFrame:
    previous = load_previous_canonical()
    latest_date = None if previous.empty else str(previous["date"].max())
    historical = pd.DataFrame() if not previous.empty and not FORCE_FULL_REBUILD else load_historical_sources()
    supabase_delta = fetch_supabase_actuals(latest_date)

    canonical = merge_priority([historical, previous, supabase_delta])
    validate_canonical(canonical)
    save_canonical(canonical)
    print(f"Canonical rows: {len(canonical):,}; latest date: {canonical['date'].max()}")
    return canonical


## Source: train

In [ ]:
from __future__ import annotations

import math
import pickle
from pathlib import Path

import numpy as np
import pandas as pd


try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None

try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor

FEATURE_COLUMNS = [
    "price_lag_1",
    "price_lag_2",
    "price_lag_3",
    "price_lag_7",
    "price_lag_14",
    "price_lag_30",
    "price_lag_60",
    "price_lag_90",
    "price_lag_180",
    "price_lag_365",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",
    "rolling_mean_60",
    "rolling_mean_90",
    "rolling_mean_180",
    "rolling_median_7",
    "rolling_median_30",
    "rolling_median_90",
    "rolling_min_30",
    "rolling_max_30",
    "rolling_min_90",
    "rolling_max_90",
    "rolling_std_7",
    "rolling_std_14",
    "rolling_std_30",
    "rolling_std_90",
    "ewm_mean_7",
    "ewm_mean_30",
    "ewm_mean_90",
    "return_1",
    "return_3",
    "return_7",
    "return_14",
    "return_30",
    "return_90",
    "momentum_7_30",
    "momentum_30_90",
    "price_vs_mean_30",
    "price_vs_mean_90",
    "price_range_pct",
    "arrival",
    "arrival_lag_1",
    "arrival_lag_7",
    "arrival_rolling_mean_7",
    "arrival_rolling_mean_30",
    "market_count",
    "market_count_lag_7",
    "market_count_rolling_mean_30",
    "month",
    "quarter",
    "day_of_week",
    "day_of_year",
    "year_index",
    "season_sin",
    "season_cos",
    "month_sin",
    "month_cos",
    "is_monsoon",
    "is_harvest_rabi",
    "is_harvest_kharif",
    "national_price",
    "national_lag_1",
    "national_lag_7",
    "national_lag_30",
    "national_lag_90",
    "national_return_7",
    "national_return_30",
    "state_national_spread",
    "state_national_ratio",
    "state_code",
]


def mape(actual: pd.Series | np.ndarray, predicted: pd.Series | np.ndarray) -> float:
    actual = pd.to_numeric(pd.Series(actual), errors="coerce")
    predicted = pd.to_numeric(pd.Series(predicted), errors="coerce")
    mask = actual.abs() > 1e-9
    if not mask.any():
        return math.inf
    return float(((predicted[mask] - actual[mask]).abs() / actual[mask].abs()).mean() * 100)


def mae(actual: pd.Series | np.ndarray, predicted: pd.Series | np.ndarray) -> float:
    actual = pd.to_numeric(pd.Series(actual), errors="coerce")
    predicted = pd.to_numeric(pd.Series(predicted), errors="coerce")
    return float((predicted - actual).abs().mean())


def fill_features(frame: pd.DataFrame, fill_values: dict[str, float]) -> pd.DataFrame:
    out = frame.reindex(columns=FEATURE_COLUMNS).copy()
    return out.fillna(fill_values).replace([np.inf, -np.inf], 0)


def feature_engineering(canonical: pd.DataFrame) -> pd.DataFrame:
    df = canonical.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["price_low"] = pd.to_numeric(df.get("price_low"), errors="coerce")
    df["price_high"] = pd.to_numeric(df.get("price_high"), errors="coerce")
    df["arrival"] = pd.to_numeric(df.get("arrival"), errors="coerce").fillna(0)
    df["market_count"] = pd.to_numeric(df["market_count"], errors="coerce").fillna(0)
    df = df.dropna(subset=["date", "state_name", "grain", "price"]).sort_values(["grain", "state_name", "date"])

    national = df[df["state_name"].eq("All States")][["date", "grain", "price"]].rename(columns={"price": "national_price"})
    df = df.merge(national, on=["date", "grain"], how="left")
    df["national_price"] = df.groupby("grain")["national_price"].ffill().bfill()

    group = df.groupby(["grain", "state_name"], sort=False)
    for lag in [1, 2, 3, 7, 14, 30, 60, 90, 180, 365]:
        df[f"price_lag_{lag}"] = group["price"].shift(lag)

    for window in [7, 14, 30, 60, 90, 180]:
        shifted = group["price"].shift(1)
        df[f"rolling_mean_{window}"] = shifted.groupby([df["grain"], df["state_name"]]).transform(lambda s: s.rolling(window, min_periods=3).mean())
        if window in {7, 30, 90}:
            df[f"rolling_median_{window}"] = shifted.groupby([df["grain"], df["state_name"]]).transform(lambda s: s.rolling(window, min_periods=3).median())
        if window in {30, 90}:
            df[f"rolling_min_{window}"] = shifted.groupby([df["grain"], df["state_name"]]).transform(lambda s: s.rolling(window, min_periods=3).min())
            df[f"rolling_max_{window}"] = shifted.groupby([df["grain"], df["state_name"]]).transform(lambda s: s.rolling(window, min_periods=3).max())
        if window in {7, 14, 30, 90}:
            df[f"rolling_std_{window}"] = shifted.groupby([df["grain"], df["state_name"]]).transform(lambda s: s.rolling(window, min_periods=3).std())

    for span in [7, 30, 90]:
        df[f"ewm_mean_{span}"] = group["price"].transform(lambda s: s.shift(1).ewm(span=span, adjust=False, min_periods=3).mean())

    for period in [1, 3, 7, 14, 30, 90]:
        df[f"return_{period}"] = group["price"].pct_change(periods=period).clip(-0.5, 0.5)

    df["momentum_7_30"] = df["rolling_mean_7"] - df["rolling_mean_30"]
    df["momentum_30_90"] = df["rolling_mean_30"] - df["rolling_mean_90"]
    df["price_vs_mean_30"] = df["price"] / df["rolling_mean_30"].replace(0, np.nan)
    df["price_vs_mean_90"] = df["price"] / df["rolling_mean_90"].replace(0, np.nan)
    df["price_range_pct"] = (df["price_high"] - df["price_low"]) / df["price"].replace(0, np.nan)

    df["arrival_lag_1"] = group["arrival"].shift(1)
    df["arrival_lag_7"] = group["arrival"].shift(7)
    df["arrival_rolling_mean_7"] = group["arrival"].transform(lambda s: s.shift(1).rolling(7, min_periods=2).mean())
    df["arrival_rolling_mean_30"] = group["arrival"].transform(lambda s: s.shift(1).rolling(30, min_periods=3).mean())
    df["market_count_lag_7"] = group["market_count"].shift(7)
    df["market_count_rolling_mean_30"] = group["market_count"].transform(lambda s: s.shift(1).rolling(30, min_periods=3).mean())

    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_year"] = df["date"].dt.dayofyear
    df["year_index"] = df["date"].dt.year - df["date"].dt.year.min()
    df["season_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["season_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["is_monsoon"] = df["month"].isin([6, 7, 8, 9]).astype(int)
    df["is_harvest_rabi"] = df["month"].isin([3, 4, 5]).astype(int)
    df["is_harvest_kharif"] = df["month"].isin([9, 10, 11]).astype(int)

    grain_group = df.groupby("grain", sort=False)
    for lag in [1, 7, 30, 90]:
        df[f"national_lag_{lag}"] = grain_group["national_price"].shift(lag)
    df["national_return_7"] = grain_group["national_price"].pct_change(periods=7).clip(-0.5, 0.5)
    df["national_return_30"] = grain_group["national_price"].pct_change(periods=30).clip(-0.5, 0.5)
    df["state_national_spread"] = df["price"] - df["national_price"]
    df["state_national_ratio"] = df["price"] / df["national_price"].replace(0, np.nan)
    df["state_code"] = df["state_name"].astype("category").cat.codes

    return df.replace([np.inf, -np.inf], np.nan)


def make_candidate_models(horizon: int, train_rows: int) -> dict[str, object]:
    models: dict[str, object] = {}

    if CatBoostRegressor is not None:
        models["catboost"] = CatBoostRegressor(
            iterations=900 if train_rows > 2000 else 500,
            depth=7,
            learning_rate=0.035,
            loss_function="RMSE",
            random_seed=42 + horizon,
            verbose=False,
            allow_writing_files=False,
            l2_leaf_reg=5,
        )

    if LGBMRegressor is not None:
        models["lightgbm"] = LGBMRegressor(
            n_estimators=900,
            learning_rate=0.035,
            num_leaves=64,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42 + horizon,
            objective="regression",
            n_jobs=-1,
        )

    if XGBRegressor is not None:
        models["xgboost"] = XGBRegressor(
            n_estimators=700,
            max_depth=6,
            learning_rate=0.035,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="reg:squarederror",
            random_state=42 + horizon,
            tree_method="hist",
            n_jobs=-1,
        )

    models["hist_gb"] = HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.035,
        l2_regularization=0.05,
        max_leaf_nodes=45,
        random_state=42 + horizon,
    )
    models["extra_trees"] = ExtraTreesRegressor(
        n_estimators=240,
        min_samples_leaf=3,
        max_features=0.85,
        random_state=42 + horizon,
        n_jobs=-1,
    )
    models["random_forest"] = RandomForestRegressor(
        n_estimators=180,
        min_samples_leaf=4,
        max_features=0.75,
        random_state=42 + horizon,
        n_jobs=-1,
    )
    return models


def tune_hist_gb(train: pd.DataFrame, valid: pd.DataFrame, feature_fill_values: dict[str, float], horizon: int) -> HistGradientBoostingRegressor | None:
    if not ENABLE_OPTUNA_TUNING or len(train) < 1000:
        return None
    try:
        import optuna
    except Exception:
        return None

    X_train = fill_features(train[FEATURE_COLUMNS], feature_fill_values)
    y_train = train["target_log_return"]
    X_valid = fill_features(valid[FEATURE_COLUMNS], feature_fill_values)
    valid_actual = valid["target_price"].to_numpy()
    valid_price = valid["price"].to_numpy()

    def objective(trial):
        model = HistGradientBoostingRegressor(
            max_iter=trial.suggest_int("max_iter", 250, 900),
            learning_rate=trial.suggest_float("learning_rate", 0.015, 0.08, log=True),
            max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 20, 80),
            l2_regularization=trial.suggest_float("l2_regularization", 1e-4, 0.5, log=True),
            random_state=42 + horizon,
        )
        model.fit(X_train, y_train)
        pred = valid_price * np.exp(model.predict(X_valid))
        return mape(valid_actual, pred)

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42 + horizon))
    study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT_SECONDS, show_progress_bar=False)
    params = study.best_params
    return HistGradientBoostingRegressor(**params, random_state=42 + horizon)


def train_candidate_models(train: pd.DataFrame, valid: pd.DataFrame, horizon: int, fill_values: dict[str, float]) -> dict[str, object]:
    fit_train = train
    if MAX_TRAIN_ROWS_PER_MODEL > 0 and len(fit_train) > MAX_TRAIN_ROWS_PER_MODEL:
        fit_train = fit_train.sample(MAX_TRAIN_ROWS_PER_MODEL, random_state=42 + horizon).sort_values("date")

    models = make_candidate_models(horizon, len(fit_train))
    tuned = tune_hist_gb(fit_train, valid, fill_values, horizon)
    if tuned is not None:
        models["optuna_hist_gb"] = tuned

    X_train = fill_features(fit_train[FEATURE_COLUMNS], fill_values)
    y_train = fit_train["target_log_return"]
    fitted = {}
    for name, model in models.items():
        try:
            model.fit(X_train, y_train)
            fitted[name] = model
        except Exception as error:
            print(f"Model skipped ({name}, {horizon}d): {type(error).__name__}: {str(error)[:120]}")
    if not fitted:
        raise RuntimeError("No candidate model could be trained")
    return fitted


def predict_candidate_prices(models: dict[str, object], frame: pd.DataFrame, fill_values: dict[str, float]) -> dict[str, np.ndarray]:
    X = fill_features(frame[FEATURE_COLUMNS], fill_values)
    prices = frame["price"].to_numpy(dtype=float)
    predictions = {"baseline": prices.copy()}
    lower = prices * 0.55
    upper = prices * 1.75
    for name, model in models.items():
        try:
            pred = prices * np.exp(model.predict(X))
            predictions[name] = np.clip(pred, lower, upper)
        except Exception as error:
            print(f"Prediction skipped ({name}): {type(error).__name__}: {str(error)[:120]}")
    return predictions


def weighted_ensemble(predictions: dict[str, np.ndarray], actual: np.ndarray) -> tuple[np.ndarray, dict[str, float]]:
    model_names = [name for name in predictions if name != "baseline"]
    if not model_names:
        return predictions["baseline"], {"baseline": 1.0}

    scores = {name: mape(actual, predictions[name]) for name in model_names}
    best_score = min(scores.values()) if scores else math.inf
    keep = [name for name in model_names if np.isfinite(scores[name]) and scores[name] <= best_score * ENSEMBLE_PRUNE_RATIO]
    if not keep:
        keep = [min(scores, key=scores.get)]

    weights_raw = np.array([1.0 / max(scores[name], 0.05) for name in keep], dtype=float)
    weights_raw = weights_raw / weights_raw.sum()
    matrix = np.vstack([predictions[name] for name in keep])
    ensemble = np.average(matrix, axis=0, weights=weights_raw)
    weights = {name: round(float(weight), 6) for name, weight in zip(keep, weights_raw)}
    return ensemble, weights


def row_payload(row: pd.Series, predicted: float, method: str) -> dict:
    return {
        "origin_date": row["date"].date().isoformat(),
        "target_date": row["target_date"].date().isoformat(),
        "state_name": row["state_name"],
        "grain": row["grain"],
        "horizon": int(row["horizon"]),
        "actual_price": float(row["target_price"]),
        "predicted_price": float(predicted),
        "method": method,
    }


def train_one(features: pd.DataFrame, grain: str, horizon: int) -> dict | None:
    data = features[features["grain"].eq(grain)].copy()
    counts = data.groupby("state_name")["date"].count()
    valid_states = counts[counts >= MIN_STATE_OBSERVED_DAYS].index
    data = data[data["state_name"].isin(valid_states)].copy()
    if data.empty:
        return None

    data["horizon"] = horizon
    data["target_date"] = data["date"] + pd.to_timedelta(horizon, unit="D")
    target = data[["state_name", "date", "price"]].rename(columns={"date": "target_date", "price": "target_price"})
    data = data.merge(target, on=["state_name", "target_date"], how="inner")
    data["target_log_return"] = np.log(data["target_price"] / data["price"])
    data = data.dropna(subset=["target_price", "target_log_return"]).sort_values("date")
    if len(data) < MIN_VALIDATION_SAMPLES * 4:
        return None

    feature_fill_values = (
        data[FEATURE_COLUMNS]
        .replace([np.inf, -np.inf], np.nan)
        .median(numeric_only=True)
        .fillna(0)
        .to_dict()
    )

    history_rows = [row_payload(row, float(row["price"]), "baseline_history") for _, row in data.iterrows()]

    cutoff = data["date"].quantile(0.8)
    train = data[data["date"] < cutoff].copy()
    valid = data[data["date"] >= cutoff].copy()
    if len(valid) < MIN_VALIDATION_SAMPLES:
        return None

    models = train_candidate_models(train, valid, horizon, feature_fill_values)
    valid_predictions = predict_candidate_prices(models, valid, feature_fill_values)
    ensemble_pred, ensemble_weights = weighted_ensemble(valid_predictions, valid["target_price"].to_numpy(dtype=float))
    valid_predictions["ensemble"] = ensemble_pred

    gates = {}
    validation_rows = []
    for state, state_valid in valid.groupby("state_name"):
        idx = state_valid.index
        if len(state_valid) < MIN_VALIDATION_SAMPLES:
            gates[state] = {"selected_method": "baseline", "reason": "insufficient_validation", "sample_count": int(len(state_valid))}
            continue

        actual = state_valid["target_price"].to_numpy(dtype=float)
        method_scores = {}
        method_mae = {}
        for method, pred_all in valid_predictions.items():
            pred = pd.Series(pred_all, index=valid.index).loc[idx].to_numpy(dtype=float)
            method_scores[method] = mape(actual, pred)
            method_mae[method] = mae(actual, pred)

        baseline_mape = method_scores.get("baseline", math.inf)
        baseline_mae = method_mae.get("baseline", math.inf)
        candidate_methods = [method for method in method_scores if method != "baseline" and np.isfinite(method_scores[method])]
        best_method = min(candidate_methods, key=lambda method: method_scores[method]) if candidate_methods else "baseline"
        selected = best_method if method_scores.get(best_method, math.inf) + MIN_MAPE_IMPROVEMENT < baseline_mape else "baseline"

        gates[state] = {
            "selected_method": selected,
            "sample_count": int(len(state_valid)),
            "baseline_mape": round(float(baseline_mape), 4) if np.isfinite(baseline_mape) else None,
            "baseline_mae": round(float(baseline_mae), 4) if np.isfinite(baseline_mae) else None,
            "ml_mape": round(float(method_scores.get(selected, baseline_mape)), 4) if np.isfinite(method_scores.get(selected, baseline_mape)) else None,
            "ml_mae": round(float(method_mae.get(selected, math.nan)), 4) if np.isfinite(method_mae.get(selected, math.nan)) else None,
            "method_mapes": {name: round(float(score), 4) for name, score in method_scores.items() if np.isfinite(score)},
            "method_maes": {name: round(float(score), 4) for name, score in method_mae.items() if np.isfinite(score)},
            "ensemble_weights": ensemble_weights,
        }

        selected_pred_all = pd.Series(valid_predictions[selected], index=valid.index)
        for row_idx, row in state_valid.iterrows():
            validation_rows.append(row_payload(row, float(selected_pred_all.loc[row_idx]), selected))

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    model_path = MODEL_DIR / f"ensemble_{grain.lower()}_{horizon}d.pkl"
    with model_path.open("wb") as handle:
        pickle.dump({"models": models, "feature_fill_values": feature_fill_values, "ensemble_weights": ensemble_weights}, handle)

    return {
        "grain": grain,
        "horizon": horizon,
        "models": models,
        "model_path": str(model_path.name),
        "feature_columns": FEATURE_COLUMNS,
        "feature_fill_values": feature_fill_values,
        "ensemble_weights": ensemble_weights,
        "gates": gates,
        "history_rows": history_rows,
        "validation_rows": validation_rows,
        "latest_training_date": data["date"].max().date().isoformat(),
    }


def train_models(canonical: pd.DataFrame) -> dict:
    features = feature_engineering(canonical)
    registry = {"features": features, "models": {}, "history_rows": [], "validation_rows": []}
    for grain in TARGET_GRAINS:
        registry["models"][grain] = {}
        for horizon in HORIZONS:
            trained = train_one(features, grain, horizon)
            if trained:
                registry["models"][grain][str(horizon)] = trained
                registry["history_rows"].extend(trained["history_rows"])
                registry["validation_rows"].extend(trained["validation_rows"])
                print(f"Trained ensemble {grain} {horizon}d ({len(trained['models'])} models)")
            else:
                print(f"Using baseline only for {grain} {horizon}d")
    return registry


## Source: predict

In [ ]:
from __future__ import annotations

import json
from datetime import timedelta

import numpy as np
import pandas as pd



def as_json_float(value: object) -> float | None:
    if pd.isna(value):
        return None
    return float(round(float(value), 4))


def interpolate_series(last_date, current_price: float, horizon_points: dict[int, float]) -> list[dict]:
    points = {0: current_price, **horizon_points}
    xs = np.array(sorted(points.keys()), dtype=float)
    ys = np.array([points[int(x)] for x in xs], dtype=float)
    max_horizon = int(xs.max()) if len(xs) else 0
    series = []
    for day in range(1, max_horizon + 1):
        price = float(np.interp(day, xs, ys))
        series.append({
            "date": (last_date + timedelta(days=day)).isoformat(),
            "price": round(price, 2),
            "is_anchor": day in horizon_points,
            "anchor_horizon": day if day in horizon_points else None,
        })
    return series


def predict_method_price(trained: dict, method: str, row: pd.Series, current_price: float) -> tuple[float, float | None]:
    if not trained or method == "baseline":
        return current_price, None

    feature_frame = row[trained["feature_columns"]].to_frame().T
    if feature_frame.isna().all(axis=None):
        return current_price, None

    models = trained.get("models") or {}
    fill_values = trained.get("feature_fill_values", {})
    features = fill_features(feature_frame, fill_values)
    lower = current_price * 0.55
    upper = current_price * 1.75

    def model_price(model_name: str) -> float | None:
        model = models.get(model_name)
        if model is None:
            return None
        value = current_price * float(np.exp(model.predict(features)[0]))
        return float(np.clip(value, lower, upper))

    if method == "ensemble":
        weights = trained.get("ensemble_weights") or {}
        weighted_values = []
        weighted_weights = []
        for model_name, weight in weights.items():
            value = model_price(model_name)
            if value is None:
                continue
            weighted_values.append(value)
            weighted_weights.append(float(weight))
        if weighted_values and sum(weighted_weights) > 0:
            price = float(np.average(np.array(weighted_values), weights=np.array(weighted_weights)))
            return price, price
        method = next(iter(models), "baseline")

    price = model_price(method)
    if price is None:
        return current_price, None
    return price, price


def generate_predictions(canonical: pd.DataFrame, registry: dict) -> tuple[dict, dict, dict, dict]:
    RELEASE_DIR.mkdir(parents=True, exist_ok=True)
    features = registry["features"].copy()
    features["date"] = pd.to_datetime(features["date"])

    predictions: dict = {}
    forecast_series: dict = {}
    actuals: dict = {}
    metrics: dict = {}

    latest_rows = features.sort_values("date").groupby(["grain", "state_name"], as_index=False).tail(1)

    for grain in TARGET_GRAINS:
        predictions[grain] = {}
        forecast_series[grain] = {}
        actuals[grain] = {}
        metrics[grain] = {}

        for _, row in latest_rows[latest_rows["grain"].eq(grain)].iterrows():
            state = row["state_name"]
            last_date = row["date"].date()
            current_price = float(row["price"])
            horizon_points: dict[int, float] = {}
            predictions[grain][state] = {
                "current_price": round(current_price, 2),
                "last_actual_date": last_date.isoformat(),
                "forecast_start_date": (last_date + timedelta(days=1)).isoformat(),
                "status": "fresh",
                "horizons": {},
            }
            metrics[grain][state] = {}

            for horizon in HORIZONS:
                trained = registry["models"].get(grain, {}).get(str(horizon))
                gate = trained["gates"].get(state) if trained else None
                selected_method = gate.get("selected_method", "baseline") if gate else "baseline"
                predicted_price, ml_price = predict_method_price(trained, selected_method, row, current_price)
                if ml_price is None and selected_method != "baseline":
                    selected_method = "baseline"

                horizon_points[horizon] = predicted_price
                metric_payload = gate or {"selected_method": selected_method, "sample_count": 0}
                metrics[grain][state][str(horizon)] = metric_payload
                predictions[grain][state]["horizons"][str(horizon)] = {
                    "target_date": (last_date + timedelta(days=horizon)).isoformat(),
                    "predicted_price": round(float(predicted_price), 2),
                    "selected_method": selected_method,
                    "metrics": {
                        "mape": as_json_float(metric_payload.get("ml_mape")),
                        "mae": as_json_float(metric_payload.get("ml_mae")),
                        "baseline_mape": as_json_float(metric_payload.get("baseline_mape")),
                        "baseline_mae": as_json_float(metric_payload.get("baseline_mae")),
                        "sample_count": int(metric_payload.get("sample_count", 0)),
                        "method_mapes": metric_payload.get("method_mapes", {}),
                        "method_maes": metric_payload.get("method_maes", {}),
                    },
                    "model_price": round(float(ml_price), 2) if ml_price is not None else None,
                }

            forecast_series[grain][state] = interpolate_series(last_date, current_price, horizon_points)

        grain_actuals = canonical[canonical["grain"].eq(grain)].copy()
        grain_actuals["date"] = pd.to_datetime(grain_actuals["date"])
        cutoff = grain_actuals["date"].max() - pd.Timedelta(days=FORECAST_HISTORY_DAYS)
        grain_actuals = grain_actuals[grain_actuals["date"].ge(cutoff)]
        for state, state_df in grain_actuals.groupby("state_name"):
            state_df = state_df.sort_values("date")
            actuals[grain][state] = {
                "context": [
                    {
                        "date": date.date().isoformat(),
                        "price": round(float(price), 2),
                        "is_observed": bool(is_observed),
                    }
                    for date, price, is_observed in zip(state_df["date"], state_df["price"], state_df["is_observed"])
                ]
            }

    (RELEASE_DIR / "predictions.json").write_text(json.dumps(predictions, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "forecast_series.json").write_text(json.dumps(forecast_series, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "actuals.json").write_text(json.dumps(actuals, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding="utf-8")
    return predictions, forecast_series, actuals, metrics


## Source: efficiency

In [ ]:
from __future__ import annotations

import json
import math

import numpy as np
import pandas as pd



def metrics_for(rows: pd.DataFrame) -> dict:
    if rows.empty:
        return {"sample_count": 0}
    error = rows["predicted_price"] - rows["actual_price"]
    pct = (error.abs() / rows["actual_price"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    return {
        "sample_count": int(len(rows)),
        "mae": round(float(error.abs().mean()), 2),
        "rmse": round(float(math.sqrt((error ** 2).mean())), 2),
        "mape": round(float(pct.mean() * 100), 2),
        "wape": round(float(error.abs().sum() / rows["actual_price"].abs().sum() * 100), 2),
    }


def generate_efficiency_data(registry: dict) -> tuple[dict, dict]:
    RELEASE_DIR.mkdir(parents=True, exist_ok=True)
    rows = pd.DataFrame([
        *registry.get("history_rows", []),
        *registry.get("validation_rows", []),
    ])
    historical_efficiency = {grain: {} for grain in TARGET_GRAINS}
    backtest = {grain: {} for grain in TARGET_GRAINS}

    if rows.empty:
        (RELEASE_DIR / "historical_efficiency.json").write_text(json.dumps(historical_efficiency, indent=2), encoding="utf-8")
        (RELEASE_DIR / "backtest.json").write_text(json.dumps(backtest, indent=2), encoding="utf-8")
        return historical_efficiency, backtest

    rows["error_pct"] = (rows["predicted_price"] - rows["actual_price"]).abs() / rows["actual_price"] * 100
    rows["method_rank"] = rows["method"].eq("baseline_history").astype(int)
    rows = (
        rows.sort_values(["grain", "state_name", "horizon", "target_date", "method_rank"])
        .drop_duplicates(["grain", "state_name", "horizon", "origin_date", "target_date"], keep="first")
        .sort_values(["grain", "state_name", "horizon", "target_date"])
    )

    for grain in TARGET_GRAINS:
        grain_rows = rows[rows["grain"].eq(grain)]
        for state, state_rows in grain_rows.groupby("state_name"):
            historical_efficiency[grain][state] = {}
            backtest[grain][state] = {}
            for horizon in HORIZONS:
                h_rows = state_rows[state_rows["horizon"].eq(horizon)]
                if EFFICIENCY_MAX_ROWS_PER_SERIES > 0:
                    h_rows = h_rows.tail(EFFICIENCY_MAX_ROWS_PER_SERIES)
                if h_rows.empty:
                    continue
                series = [
                    {
                        "origin_date": row.origin_date,
                        "date": row.target_date,
                        "actual_price": round(float(row.actual_price), 2),
                        "predicted_price": round(float(row.predicted_price), 2),
                        "error_pct": round(float(row.error_pct), 2),
                        "method": row.method,
                    }
                    for row in h_rows.itertuples()
                ]
                payload = {
                    "horizon_days": horizon,
                    "metrics": metrics_for(h_rows),
                    "series": series,
                }
                historical_efficiency[grain][state][str(horizon)] = payload
                backtest[grain][state][str(horizon)] = {
                    "backtestDate": h_rows["target_date"].max(),
                    "metrics": payload["metrics"],
                    "comparisons": [
                        {
                            "date": item["date"],
                            "actualPrice": item["actual_price"],
                            "predictedPrice": item["predicted_price"],
                            "difference": round(item["predicted_price"] - item["actual_price"], 2),
                            "errorPct": item["error_pct"],
                        }
                        for item in series[-15:]
                    ],
                }

    (RELEASE_DIR / "historical_efficiency.json").write_text(json.dumps(historical_efficiency, indent=2, allow_nan=False), encoding="utf-8")
    (RELEASE_DIR / "backtest.json").write_text(json.dumps(backtest, indent=2, allow_nan=False), encoding="utf-8")
    return historical_efficiency, backtest


## Source: reasoning

In [ ]:
from __future__ import annotations

import json



def generate_reasoning(predictions: dict, metrics: dict) -> dict:
    reasoning = {grain: {} for grain in TARGET_GRAINS}
    for grain, states in predictions.items():
        for state, payload in states.items():
            current = payload.get("current_price") or 0
            reasoning[grain][state] = {}
            for horizon in HORIZONS:
                horizon_key = str(horizon)
                h_payload = payload.get("horizons", {}).get(horizon_key)
                if not h_payload:
                    continue
                predicted = h_payload["predicted_price"]
                method = h_payload.get("selected_method", "baseline")
                change_pct = ((predicted - current) / current * 100) if current else 0
                direction = "rise" if change_pct >= 0 else "fall"
                state_metrics = metrics.get(grain, {}).get(state, {}).get(horizon_key, {})
                sample_count = state_metrics.get("sample_count", 0)
                baseline_mape = state_metrics.get("baseline_mape")
                ml_mape = state_metrics.get("ml_mape")
                drivers = [
                    {"feature": "recent_state_price_momentum", "score": 0.35},
                    {"feature": "national_price_relationship", "score": 0.3},
                    {"feature": "seasonality", "score": 0.2},
                    {"feature": "market_count_signal", "score": 0.15},
                ]
                text = (
                    f"For {state}, {grain} is projected to {direction} by "
                    f"{abs(change_pct):.1f}% over {horizon} days. The selected method is "
                    f"{method}, based on the state-specific validation gate."
                )
                if sample_count:
                    text += f" The gate used {sample_count} validation samples."
                if ml_mape is not None and baseline_mape is not None:
                    text += f" ML validation MAPE was {ml_mape:.2f}% versus persistence baseline {baseline_mape:.2f}%."
                reasoning[grain][state][horizon_key] = {
                    "source": "rule_based",
                    "text": text,
                    "key_drivers": drivers,
                }
    (RELEASE_DIR / "reasoning.json").write_text(json.dumps(reasoning, indent=2, allow_nan=False), encoding="utf-8")
    return reasoning


## Source: manifest

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import uuid
import zipfile
from datetime import datetime, timezone

import pandas as pd



def sha256(path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_states(canonical: pd.DataFrame) -> list[str]:
    states = sorted(canonical["state_name"].dropna().unique().tolist(), key=lambda state: (state != "All States", state))
    payload = {
        "states": [
            {
                "state_name": state,
                "state_key": "all-states" if state == "All States" else state.lower().replace(" ", "-"),
            }
            for state in states
        ]
    }
    (RELEASE_DIR / "states.json").write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")
    return states


def mirror_release_for_kaggle_output() -> None:
    files = [path for path in sorted(RELEASE_DIR.iterdir()) if path.is_file()]
    zip_path = WORK_ROOT / "grainology_release.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            shutil.copy(path, WORK_ROOT / path.name)
            archive.write(path, arcname=f"release/{path.name}")

    print(f"Mirrored {len(files)} release files to {WORK_ROOT}")
    print(f"Release archive written to {zip_path}")


def finalize_release(canonical: pd.DataFrame) -> dict:
    RELEASE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(CANONICAL_PARQUET, RELEASE_DIR / "canonical_daily.parquet")
    shutil.copy(CANONICAL_CSV, RELEASE_DIR / "canonical_daily.csv")
    states = write_states(canonical)

    files = {
        path.name: sha256(path)
        for path in RELEASE_DIR.iterdir()
        if path.is_file() and path.name not in {"manifest.json", "checksums.json"}
    }
    checksums = dict(sorted(files.items()))
    (RELEASE_DIR / "checksums.json").write_text(json.dumps(checksums, indent=2), encoding="utf-8")
    files["checksums.json"] = sha256(RELEASE_DIR / "checksums.json")

    run_id = os.environ.get("KAGGLE_KERNEL_RUN_ID")
    try:
        run_id = str(uuid.UUID(str(run_id)))
    except Exception:
        run_id = str(uuid.uuid4())

    generated_at = datetime.now(timezone.utc).isoformat()
    data_latest_date = pd.to_datetime(canonical["date"]).max().date().isoformat()
    manifest = {
        "schema_version": RELEASE_SCHEMA_VERSION,
        "canonical_schema_version": CANONICAL_SCHEMA_VERSION,
        "run_id": run_id,
        "status": "success",
        "generated_at": generated_at,
        "data_latest_date": data_latest_date,
        "actuals_max_updated_at": generated_at,
        "actuals_row_count": int(len(canonical)),
        "forecast_start_date": (pd.to_datetime(data_latest_date) + pd.Timedelta(days=1)).date().isoformat(),
        "model_mode": MODEL_MODE,
        "grains": TARGET_GRAINS,
        "horizons": HORIZONS,
        "states": states,
        "aggregation_method": os.environ.get("AGGREGATION_METHOD", "median"),
        "code_version": os.environ.get("GITHUB_SHA") or os.environ.get("KAGGLE_KERNEL_RUN_ID") or "kaggle-local",
        "files": dict(sorted(files.items())),
    }
    (RELEASE_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding="utf-8")

    missing = [name for name in REQUIRED_RELEASE_FILES if not (RELEASE_DIR / name).exists()]
    if missing:
        raise ValueError(f"Release bundle missing required files: {missing}")

    mirror_release_for_kaggle_output()
    print(f"Release finalized at {RELEASE_DIR}")
    print(f"Staging source was {STAGING_DIR}")
    return manifest


## Source: diagnostics

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None



def display_frame(frame: pd.DataFrame, rows: int = 20):
    try:
        from IPython.display import display
        display(frame.head(rows))
    except Exception:
        print(frame.head(rows).to_string(index=False))


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def canonical_summary(canonical: pd.DataFrame) -> pd.DataFrame:
    print_section("Canonical Dataset Summary")
    frame = canonical.copy()
    frame["date"] = pd.to_datetime(frame["date"])
    summary = pd.DataFrame([{
        "rows": len(frame),
        "min_date": frame["date"].min().date().isoformat(),
        "max_date": frame["date"].max().date().isoformat(),
        "grains": frame["grain"].nunique(),
        "states": frame["state_name"].nunique(),
        "observed_rows": int(frame["is_observed"].fillna(False).sum()) if "is_observed" in frame else None,
    }])
    display_frame(summary)

    by_grain = (
        frame.groupby("grain")
        .agg(rows=("price", "size"), min_date=("date", "min"), max_date=("date", "max"), states=("state_name", "nunique"), avg_price=("price", "mean"))
        .reset_index()
    )
    by_grain["min_date"] = by_grain["min_date"].dt.date.astype(str)
    by_grain["max_date"] = by_grain["max_date"].dt.date.astype(str)
    by_grain["avg_price"] = by_grain["avg_price"].round(2)
    print("\nRows by grain")
    display_frame(by_grain, rows=50)

    latest_date = frame["date"].max()
    latest_rows = frame[frame["date"].eq(latest_date)].groupby("grain").agg(rows=("price", "size"), states=("state_name", "nunique")).reset_index()
    print(f"\nLatest canonical date: {latest_date.date().isoformat()}")
    display_frame(latest_rows, rows=50)
    return by_grain


def recent_data_summary(canonical: pd.DataFrame, days: int = 14) -> pd.DataFrame:
    print_section(f"Recently Added / Latest {days} Days")
    frame = canonical.copy()
    frame["date"] = pd.to_datetime(frame["date"])
    cutoff = frame["date"].max() - pd.Timedelta(days=days - 1)
    recent = frame[frame["date"].ge(cutoff)]
    summary = (
        recent.groupby(["date", "grain"])
        .agg(rows=("price", "size"), states=("state_name", "nunique"), avg_price=("price", "mean"), observed=("is_observed", "sum"))
        .reset_index()
        .sort_values(["date", "grain"], ascending=[False, True])
    )
    summary["date"] = summary["date"].dt.date.astype(str)
    summary["avg_price"] = summary["avg_price"].round(2)
    display_frame(summary, rows=days * len(TARGET_GRAINS))
    return summary


def missingness_summary(canonical: pd.DataFrame) -> pd.DataFrame:
    print_section("Missingness / Data Quality")
    cols = ["price", "price_low", "price_high", "arrival", "market_count", "source", "is_observed"]
    available = [col for col in cols if col in canonical.columns]
    summary = pd.DataFrame({
        "column": available,
        "missing_pct": [round(float(canonical[col].isna().mean() * 100), 2) for col in available],
        "non_null": [int(canonical[col].notna().sum()) for col in available],
    })
    display_frame(summary, rows=50)
    duplicate_count = int(canonical.duplicated(["date", "state_name", "grain"]).sum())
    print(f"Duplicate date/state/grain rows: {duplicate_count}")
    return summary


def plot_history(canonical: pd.DataFrame, states: list[str] | None = None, grains: list[str] | None = None) -> None:
    print_section("Historical Price Trend")
    if plt is None:
        print("matplotlib is not available; skipping plots.")
        return
    frame = canonical.copy()
    frame["date"] = pd.to_datetime(frame["date"])
    states = states or ["All States"]
    grains = grains or TARGET_GRAINS
    plot_df = frame[frame["state_name"].isin(states) & frame["grain"].isin(grains)].sort_values("date")
    if plot_df.empty:
        print("No rows available for requested history plot.")
        return
    fig, axes = plt.subplots(len(grains), 1, figsize=(14, 3.2 * len(grains)), sharex=True)
    axes = np.atleast_1d(axes)
    for axis, grain in zip(axes, grains):
        grain_df = plot_df[plot_df["grain"].eq(grain)]
        for state, state_df in grain_df.groupby("state_name"):
            axis.plot(state_df["date"], state_df["price"], linewidth=1.4, label=state)
        axis.set_title(f"{grain} price trend")
        axis.set_ylabel("Price")
        axis.grid(alpha=0.25)
        axis.legend(loc="upper left")
    plt.tight_layout()
    plt.show()


def plot_recent_history(canonical: pd.DataFrame, days: int = 365, state: str = "All States") -> None:
    print_section(f"Recent {days}-Day Trend: {state}")
    if plt is None:
        print("matplotlib is not available; skipping plots.")
        return
    frame = canonical.copy()
    frame["date"] = pd.to_datetime(frame["date"])
    cutoff = frame["date"].max() - pd.Timedelta(days=days)
    frame = frame[frame["date"].ge(cutoff) & frame["state_name"].eq(state)]
    if frame.empty:
        print("No recent rows available.")
        return
    pivot = frame.pivot_table(index="date", columns="grain", values="price", aggfunc="median").sort_index()
    pivot.plot(figsize=(14, 5), linewidth=1.7, title=f"{state}: recent price trend")
    plt.grid(alpha=0.25)
    plt.ylabel("Price")
    plt.tight_layout()
    plt.show()


def training_summary(registry: dict) -> pd.DataFrame:
    print_section("Training / Model Selection Summary")
    rows = []
    for grain, horizons in registry.get("models", {}).items():
        for horizon, trained in horizons.items():
            gates = trained.get("gates", {})
            method_counts = pd.Series([gate.get("selected_method", "unknown") for gate in gates.values()]).value_counts().to_dict()
            mapes = [
                gate.get("ml_mape")
                for gate in gates.values()
                if gate.get("ml_mape") is not None and np.isfinite(gate.get("ml_mape"))
            ]
            rows.append({
                "grain": grain,
                "horizon": int(horizon),
                "candidate_models": ", ".join(trained.get("models", {}).keys()),
                "states": len(gates),
                "method_counts": method_counts,
                "median_selected_mape": round(float(np.median(mapes)), 3) if mapes else None,
                "mean_selected_mape": round(float(np.mean(mapes)), 3) if mapes else None,
            })
    summary = pd.DataFrame(rows).sort_values(["grain", "horizon"]) if rows else pd.DataFrame()
    display_frame(summary, rows=100)
    return summary


def method_leaderboard(registry: dict, top: int = 40) -> pd.DataFrame:
    print_section("Per-State Method Leaderboard")
    rows = []
    for grain, horizons in registry.get("models", {}).items():
        for horizon, trained in horizons.items():
            for state, gate in trained.get("gates", {}).items():
                method_mapes = gate.get("method_mapes") or {}
                best_method = min(method_mapes, key=method_mapes.get) if method_mapes else gate.get("selected_method")
                rows.append({
                    "grain": grain,
                    "horizon": int(horizon),
                    "state": state,
                    "selected_method": gate.get("selected_method"),
                    "best_method": best_method,
                    "selected_mape": gate.get("ml_mape"),
                    "baseline_mape": gate.get("baseline_mape"),
                    "sample_count": gate.get("sample_count"),
                    "all_method_mapes": method_mapes,
                })
    leaderboard = pd.DataFrame(rows)
    if leaderboard.empty:
        print("No leaderboard rows available.")
        return leaderboard
    leaderboard = leaderboard.sort_values(["grain", "horizon", "selected_mape"], na_position="last")
    display_frame(leaderboard, rows=top)
    return leaderboard


def validation_rows_summary(registry: dict) -> pd.DataFrame:
    print_section("Validation Rows Summary")
    rows = pd.DataFrame(registry.get("validation_rows", []))
    if rows.empty:
        print("No validation rows available.")
        return rows
    rows["error_pct"] = (rows["predicted_price"] - rows["actual_price"]).abs() / rows["actual_price"].replace(0, np.nan) * 100
    rows["abs_error"] = (rows["predicted_price"] - rows["actual_price"]).abs()
    summary = (
        rows.groupby(["grain", "horizon", "method"])
        .agg(rows=("actual_price", "size"), mape=("error_pct", "mean"), mae=("abs_error", "mean"))
        .reset_index()
    )
    summary["mape"] = summary["mape"].round(3)
    summary["mae"] = summary["mae"].round(2)
    display_frame(summary.sort_values(["grain", "horizon", "mape"]), rows=100)
    return rows


def plot_validation_fit(registry: dict, grain: str = "Wheat", state: str = "All States", horizon: int = 7) -> None:
    print_section(f"Validation Fit Plot: {grain} / {state} / {horizon}d")
    if plt is None:
        print("matplotlib is not available; skipping plots.")
        return
    rows = pd.DataFrame(registry.get("validation_rows", []))
    if rows.empty:
        print("No validation rows available.")
        return
    rows = rows[
        rows["grain"].eq(grain)
        & rows["state_name"].eq(state)
        & rows["horizon"].eq(horizon)
    ].copy()
    if rows.empty:
        print("No matching validation rows.")
        return
    rows["target_date"] = pd.to_datetime(rows["target_date"])
    rows = rows.sort_values("target_date")
    plt.figure(figsize=(14, 5))
    plt.plot(rows["target_date"], rows["actual_price"], label="Actual", linewidth=2)
    plt.plot(rows["target_date"], rows["predicted_price"], label="Predicted", linewidth=1.7)
    plt.title(f"{grain} {state} {horizon}d validation fit")
    plt.ylabel("Price")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


def efficiency_summary(efficiency: dict) -> pd.DataFrame:
    print_section("Historical Efficiency / Backtest Summary")
    rows = []
    for grain, states in efficiency.items():
        for state, horizons in states.items():
            for horizon, payload in horizons.items():
                metric = payload.get("metrics", {})
                rows.append({
                    "grain": grain,
                    "state": state,
                    "horizon": int(horizon),
                    "sample_count": metric.get("sample_count"),
                    "mape": metric.get("mape"),
                    "mae": metric.get("mae"),
                    "rmse": metric.get("rmse"),
                    "wape": metric.get("wape"),
                })
    summary = pd.DataFrame(rows)
    if summary.empty:
        print("No efficiency rows available.")
        return summary
    display_frame(summary.sort_values(["grain", "horizon", "mape"], na_position="last"), rows=100)
    return summary


def plot_efficiency_series(efficiency: dict, grain: str = "Wheat", state: str = "All States", horizon: int = 7, tail: int = 365) -> None:
    print_section(f"Efficiency Series Plot: {grain} / {state} / {horizon}d")
    if plt is None:
        print("matplotlib is not available; skipping plots.")
        return
    payload = efficiency.get(grain, {}).get(state, {}).get(str(horizon), {})
    series = pd.DataFrame(payload.get("series", []))
    if series.empty:
        print("No efficiency series for this selection.")
        return
    series["date"] = pd.to_datetime(series["date"])
    series = series.sort_values("date").tail(tail)
    plt.figure(figsize=(14, 5))
    plt.plot(series["date"], series["actual_price"], label="Actual", linewidth=2)
    plt.plot(series["date"], series["predicted_price"], label="Predicted", linewidth=1.5)
    plt.title(f"{grain} {state} {horizon}d efficiency, last {tail} rows")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


def release_file_summary() -> pd.DataFrame:
    print_section("Release File Validation")
    files = sorted(path for path in RELEASE_DIR.iterdir() if path.is_file())
    summary = pd.DataFrame([{
        "file": path.name,
        "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
    } for path in files])
    display_frame(summary, rows=100)

    manifest_path = RELEASE_DIR / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        print("\nManifest core:")
        display_frame(pd.DataFrame([{
            "run_id": manifest.get("run_id"),
            "generated_at": manifest.get("generated_at"),
            "data_latest_date": manifest.get("data_latest_date"),
            "actuals_row_count": manifest.get("actuals_row_count"),
            "states": len(manifest.get("states", [])),
            "grains": ", ".join(manifest.get("grains", [])),
        }]))
    return summary


def inspect_prediction_output(grain: str = "Wheat", state: str = "All States") -> dict:
    print_section(f"Prediction Output Inspection: {grain} / {state}")
    path = RELEASE_DIR / "predictions.json"
    if not path.exists():
        print("predictions.json not found.")
        return {}
    predictions = json.loads(path.read_text(encoding="utf-8"))
    payload = predictions.get(grain, {}).get(state, {})
    print(json.dumps(payload, indent=2)[:4000])
    return payload


## 1. Build Canonical Dataset

Loads historical Kaggle datasets, merges latest Supabase/cache data when configured, normalizes states/grains, and writes the canonical daily dataset.


In [ ]:
canonical = build_canonical_dataset()
canonical_summary(canonical)
missingness_summary(canonical)
recent_data_summary(canonical, days=14)


In [ ]:
plot_history(canonical, states=["All States"], grains=["Wheat", "Paddy", "Maize", "Mustard"])
plot_recent_history(canonical, days=365, state="All States")


## 2. Train State-aware Ensemble Models

Creates lag/rolling/arrival/seasonality/national-spread features, trains candidate models for every grain and horizon, prunes weak models, and selects the best method per state.


In [ ]:
registry = train_models(canonical)
training_summary(registry)
method_leaderboard(registry, top=80)
validation_rows_summary(registry)


In [ ]:
plot_validation_fit(registry, grain="Wheat", state="All States", horizon=7)
plot_validation_fit(registry, grain="Wheat", state="All States", horizon=30)
plot_validation_fit(registry, grain="Wheat", state="All States", horizon=90)


## 3. Generate Forecasts, Actual Context, And Dashboard Metrics

Writes `predictions.json`, `forecast_series.json`, `actuals.json`, and `metrics.json` in the website schema.


In [ ]:
predictions, forecast_series, actuals, metrics = generate_predictions(canonical, registry)
print("Prediction grains:", list(predictions.keys()))
inspect_prediction_output("Wheat", "All States")


## 4. Generate Historical Efficiency And Backtests

Writes state-wise predicted-vs-actual comparison rows for the dashboard chart and table.


In [ ]:
efficiency, backtest = generate_efficiency_data(registry)
efficiency_summary(efficiency)


In [ ]:
plot_efficiency_series(efficiency, grain="Wheat", state="All States", horizon=7, tail=365)
plot_efficiency_series(efficiency, grain="Wheat", state="All States", horizon=30, tail=365)
plot_efficiency_series(efficiency, grain="Wheat", state="All States", horizon=90, tail=365)


## 5. Generate Reasoning Text

Writes the human-readable explanation layer consumed by the website.


In [ ]:
reasoning = generate_reasoning(predictions, metrics)
print("Reasoning generated for grains:", list(reasoning.keys()))
print(reasoning.get("Wheat", {}).get("All States", {}))


## 6. Finalize Release Bundle

Validates required files, writes `manifest.json`, mirrors files to `/kaggle/working`, and creates the downloadable release archive.


In [ ]:
manifest = finalize_release(canonical)
release_file_summary()
manifest
